## 第三章学习总结：神经网络与反向传播算法
一、本章学习目标

本章主要学习了神经网络的基本结构、前向传播、反向传播算法（Backpropagation，BP）、梯度检查，以及神经网络的实际应用。

与前两章的感知器和线性单元相比，这一章最大的变化是：不再只使用一个神经元，而是把多个神经元按照一定规则连接起来，形成能够学习复杂非线性关系的神经网络。

我分别使用普通 Python 和 NumPy 实现了全连接神经网络，也使用自己编写的 BP 网络完成了 MNIST 手写数字识别实验。

## 神经网络的基本结构

神经网络由多个神经元组成，通常包含三个部分：

输入层：接收外部输入的数据。
隐藏层：对输入数据进行计算和特征转换。
输出层：给出最终预测结果。

在全连接神经网络中，相邻两层之间的神经元两两连接，每条连接都有一个权重，同一层内的神经元之间没有连接。

本章使用 sigmoid 作为神经元的激活函数：

sigmoid(x) = 1 / (1 + e^(-x))

它的输出范围是 0～1，并且具有一个很重要的性质：

sigmoid'(x) = sigmoid(x) × (1 - sigmoid(x))

这使得反向传播时可以直接利用神经元的输出计算导数。

## 前向传播

前向传播（Forward Propagation）就是将输入数据逐层传递，最终计算出网络预测结果的过程。

对于一个神经元：

y = sigmoid(w1×x1 + w2×x2 + ... + b)

其中，x 是输入，w 是权重，b 是偏置，y 是输出。

对于一整层神经元，可以把计算写成矩阵形式：

a = sigmoid(Wx + b)

前向传播的执行过程是：

输入数据 → 计算隐藏层输出 → 计算输出层输出 → 得到预测结果。

我理解到，神经网络本质上就是将多次线性变换和非线性激活函数组合在一起，建立输入与输出之间的映射关系。

## 反向传播算法（BP）

反向传播是本章最重要、也是我花时间理解最多的内容。

神经网络训练的目标，是通过调整权重和偏置，使预测结果尽可能接近真实答案。

对于一个样本，可以使用平方误差作为损失函数：

E = 1/2 × Σ(t_i - y_i)²

其中：

t 是真实目标值。
y 是神经网络预测值。
E 是损失。

反向传播通过链式求导，计算每个神经元及每条连接对损失的影响。

可以说误差项应该是困扰我最久的一个东西，可以是从数学来说看作损失函数对加权输入的影响，所以还是要用到链式法则，相当于两个部分求导。当然还分为输出层和隐藏层。隐藏层主要是看后面的节点往前的反向传播

1. 输出层的误差项

使用 sigmoid 激活函数时：

δ_j = (t_j - y_j) × y_j × (1 - y_j)

其中：

t_j - y_j 表示目标值与预测值的差。
y_j × (1 - y_j) 是 sigmoid 的导数。
δ_j 表示输出节点的误差项。

需要特别注意：delta 不等同于简单的预测误差。

在本章的符号约定中，节点的 delta 定义为损失对该节点加权输入的负偏导数。

2. 隐藏层的误差项

隐藏层没有直接对应的真实目标值，因此不能使用 t - y 计算。

隐藏层需要依靠下游节点传回来的误差信号：

δ_j = a_j × (1 - a_j) × Σ(w_kj × δ_k)

这表示，一个隐藏层节点的误差项由两部分决定：

一是自身激活函数的导数，二是它连接的下游节点的 delta 与对应权重的加权和。

我对反向传播最重要的理解是：

先计算输出层的 delta，再一层一层向前计算隐藏层的 delta，因为前一层需要利用后一层已经算好的结果。

3. 权重和偏置更新

计算出 delta 后，就可以使用梯度下降调整权重：

w_ji = w_ji + η × δ_j × x_i

其中：

η 是学习率。
δ_j 是下游节点的误差项。
x_i 是上游节点的输出。

偏置可以理解为一个输出始终为 1 的特殊输入节点，所以它也能采用相同的更新思路。

最终，我把一次训练总结为三个步骤：

前向传播 → 反向计算 delta → 更新权重。

## 手写 BP 神经网络

为了理解反向传播的执行过程，我首先使用普通 Python 实现了神经网络。

代码主要包含 Node、ConstNode、Connection、Layer、Connections 和 Network 等类。

其中：

Node 负责保存节点输出、计算 delta；ConstNode 用来实现偏置；Connection 保存连接权重并负责更新；Layer 管理一层的节点；Network 负责将各层连接起来并执行预测和训练。

核心训练过程是：

predict(sample) → calc_delta(label) → update_weight(rate)

通过这个版本，我理解了神经元的输出怎样传给下一层，以及误差信号怎样沿着连接反向传递。

在手敲代码时，我还遇到了几类实际问题，例如将 __init__ 写错、将 calc_hidden_layer_delta 拼写成 clac_hidden_layer_delta，以及数据文件路径不正确导致 FileNotFoundError。

这些调试经历让我认识到：理解公式是一方面，准确实现类、方法和数据流同样重要。

梯度检查

反向传播代码能够运行，并不一定代表计算出来的梯度正确。

本章介绍了梯度检查（Gradient Checking），通过数值方法近似计算梯度，并与反向传播的结果进行比较。

数值梯度近似公式为：

∂E/∂w ≈ [E(w + ε) - E(w - ε)] / (2ε)

这里的 ε 是一个非常小的数，例如 0.0001。

由于我们代码中的 Connection.gradient 使用的是损失函数梯度的相反数，因此在比较时需要注意符号方向。

梯度检查的意义是：用一种独立于反向传播的计算方式，验证自己实现的梯度是否正确。

这对于神经网络调试非常重要。

## MNIST 手写数字识别

本章使用 MNIST 数据集验证神经网络的实际效果。

MNIST 图片的大小是 28×28，因此每张图片可以转换成包含 784 个像素值的一维输入向量。

因为需要识别数字 0～9，输出层包含 10 个神经元。

原教程选择的网络结构是：

Network([784, 300, 10])

标签使用 10 维向量表示：正确数字对应的位置为 0.9，其他位置为 0.1。

预测时，取输出向量中最大值对应的索引，作为识别结果。

我的实验结果

为了缩短初次训练时间，我先使用：

200 张训练图片、50 张测试图片。
20 个隐藏层节点。
学习率 0.3，训练 3 轮。

结果如下：

训练轮次	错误率	准确率
第 1 轮	90%	10%
第 2 轮	66%	34%
第 3 轮	40%	60%

第三轮测试时，50 张图片中有 30 张识别正确。

第一张测试图片的真实数字为 7，网络预测结果也是 7。

通过这次实验，我直观地看到了：随着反向传播不断调整权重，网络在测试图片上的识别准确率有所提高。

当然，由于训练和测试样本都比较少，60% 还不能代表模型在整个 MNIST 数据集上的准确率。

八、向量化编程

在完成普通 Python 版本后，本章进一步介绍了向量化编程。

原来的实现需要为每个神经元建立 Node 对象，为每条连接建立 Connection 对象，并通过 Python 循环完成计算。

向量化编程使用 NumPy 数组，将多个神经元的计算合并成矩阵运算。

核心公式为：

前向传播：

a = sigmoid(Wx + b)

权重调整信号：

W_grad = δ × x.T

更新权重：

W = W + η × W_grad

更新偏置：

b = b + η × δ

在代码中，最具有代表性的操作是：

np.dot(W, x) + b

它能代替大量逐条连接的计算。

新的实现主要使用 SigmoidActivator、FullConnectedLayer 和 Network 三个类，不再需要为每个神经元和连接分别建立对象。

我也理解了 NumPy 矩阵形状的重要性。例如，784 个输入需要整理为 (784, 1) 的列向量，10 个目标值要整理为 (10, 1)。

向量化并没有改变反向传播的数学原理，而是让计算方式更加简洁，也能利用 NumPy 底层经过优化的矩阵运算提高执行效率。

In [9]:
from math import exp
import random

def sigmoid(x):
    return 1.0/(1.0+exp(-x))

class Node:
    def __init__(self, layer_index, node_index):
        self.layer_index = layer_index
        self.node_index = node_index
        self.downstream = []
        self.upstream = []
        self.output = 0.0
        self.delta = 0.0
    def set_output(self, output):
        self.output = output
    def append_downstream_connection(self, conn):
        self.downstream.append(conn)
    def append_upstream_connection(self, conn):
        self.upstream.append(conn)
    def calc_output(self):
        weighted_sum=sum(
            conn.upstream_node.output * conn.weight
            for conn in self.upstream
        )
        self.output = sigmoid(weighted_sum)
    def calc_output_layer_delta(self, label):
        self.delta = (
            self.output
            *(1-self.output)
            *(label-self.output)
        )
    
    def calc_hidden_layer_delta(self):
        downstream_delta = sum(
            conn.downstream_node.delta * conn.weight
            for conn in self.downstream
        )

        self.delta = (
            self.output
            * (1 - self.output)
            * downstream_delta
        )
class ConstNode:
    def __init__(self, layer_index, node_index):
        self.layer_index = layer_index
        self.node_index = node_index
        self.downstream = []
        self.output = 1.0
        self.delta = 0.0
    def append_downstream_connection(self, conn):
        self.downstream.append(conn)

    def calc_hidden_layer_delta(self):
        downstream_delta = sum(
            conn.downstream_node.delta * conn.weight
            for conn in self.downstream
        )

        self.delta = (
            self.output
            * (1 - self.output)
            * downstream_delta
        )

class Connection:
    def __init__(self, upstream_node, downstream_node):
        self.upstream_node = upstream_node
        self.downstream_node = downstream_node

        self.weight = random.uniform(-0.1, 0.1)
        self.gradient = 0.0

    def calc_gradient(self):
        self.gradient = (
            self.downstream_node.delta
            * self.upstream_node.output
        )

    def get_gradient(self):
        return self.gradient

    def update_weight(self, rate):
        self.calc_gradient()
        self.weight += rate * self.gradient

    def __str__(self):
        return (
            f"({self.upstream_node.layer_index}-"
            f"{self.upstream_node.node_index}) -> "
            f"({self.downstream_node.layer_index}-"
            f"{self.downstream_node.node_index}) = "
            f"{self.weight:.6f}"
        )

class Layer:
    def __init__(self, layer_index, node_count):
        self.layer_index = layer_index
        self.nodes = []

        for i in range(node_count):
            self.nodes.append(Node(layer_index, i))

        self.nodes.append(
            ConstNode(layer_index, node_count)
        )

    def set_output(self, data):
        for i in range(len(data)):
            self.nodes[i].set_output(data[i])

    def calc_output(self):
        for node in self.nodes[:-1]:
            node.calc_output()

    def dump(self):
        for node in self.nodes:
            print(
                f"{node.layer_index}-{node.node_index}: "
                f"output={node.output}, delta={node.delta}"
            )

class Connections:
    def __init__(self):
        self.connections = []

    def add_connection(self, connection):
        self.connections.append(connection)

    def dump(self):
        for conn in self.connections:
            print(conn)


class Network:
    def __init__(self, layers):
        self.connections = Connections()
        self.layers = []

        layer_count = len(layers)

        for i in range(layer_count):
            self.layers.append(Layer(i, layers[i]))

        for layer in range(layer_count - 1):
            for upstream_node in self.layers[layer].nodes:
                for downstream_node in self.layers[layer + 1].nodes[:-1]:
                    conn = Connection(
                        upstream_node,
                        downstream_node
                    )

                    self.connections.add_connection(conn)

                    upstream_node.append_downstream_connection(conn)
                    downstream_node.append_upstream_connection(conn)

    def predict(self, sample):
        self.layers[0].set_output(sample)

        for i in range(1, len(self.layers)):
            self.layers[i].calc_output()

        return [
            node.output
            for node in self.layers[-1].nodes[:-1]
        ]
    
    def calc_delta(self, label):
        output_nodes = self.layers[-1].nodes

        for i in range(len(label)):
            output_nodes[i].calc_output_layer_delta(label[i])

        for layer in self.layers[-2::-1]:
            for node in layer.nodes:
                node.calc_hidden_layer_delta()

    def update_weight(self, rate):
        for layer in self.layers[:-1]:
            for node in layer.nodes:
                for conn in node.downstream:
                    conn.update_weight(rate)

    def train_one_sample(self, label, sample, rate):
        self.predict(sample)
        self.calc_delta(label)
        self.update_weight(rate)

    def train(self, labels, data_set, rate, iteration):
        for i in range(iteration):
            for d in range(len(data_set)):
                self.train_one_sample(
                    labels[d],
                    data_set[d],
                    rate
                )
    
    def calc_gradient(self):
        for layer in self.layers[:-1]:
            for node in layer.nodes:
                for conn in node.downstream:
                    conn.calc_gradient()

    def get_gradient(self, label, sample):
        self.predict(sample)
        self.calc_delta(label)
        self.calc_gradient()

def network_error(output, label):
    return 0.5 * sum(
        (y - t) ** 2
        for y, t in zip(output, label)
    )

def gradient_check(network, sample_feature, sample_label):
    network.get_gradient(sample_label, sample_feature)

    epsilon = 0.0001
    max_diff = 0.0

    for conn in network.connections.connections:
        actual_gradient = conn.get_gradient()

        original_weight = conn.weight

        conn.weight = original_weight + epsilon

        error1 = network_error(
            network.predict(sample_feature),
            sample_label
        )

        conn.weight = original_weight - epsilon

        error2 = network_error(
            network.predict(sample_feature),
            sample_label
        )

        conn.weight = original_weight

        expected_gradient = (
            (error2 - error1) / (2 * epsilon)
        )

        diff = abs(expected_gradient - actual_gradient)
        max_diff = max(max_diff, diff)

        print(
            f"数值梯度: {expected_gradient:.8f}, "
            f"反向传播: {actual_gradient:.8f}, "
            f"差值: {diff:.10f}"
        )

    print("最大差值:", max_diff)
    return max_diff

if __name__ == "__main__":
    network = Network([2, 3, 1])

    for conn in network.connections.connections:
        conn.weight = 0.1

    sample = [0.6, 0.2]
    label = [1.0]

    max_diff = gradient_check(
        network,
        sample,
        label
    )

    assert max_diff < 1e-6

    print("梯度检查通过！")



             

数值梯度: 0.00158857, 反向传播: 0.00158857, 差值: 0.0000000000
数值梯度: 0.00158857, 反向传播: 0.00158857, 差值: 0.0000000000
数值梯度: 0.00158857, 反向传播: 0.00158857, 差值: 0.0000000000
数值梯度: 0.00052952, 反向传播: 0.00052952, 差值: 0.0000000000
数值梯度: 0.00052952, 反向传播: 0.00052952, 差值: 0.0000000000
数值梯度: 0.00052952, 反向传播: 0.00052952, 差值: 0.0000000000
数值梯度: 0.00264761, 反向传播: 0.00264761, 差值: 0.0000000000
数值梯度: 0.00264761, 反向传播: 0.00264761, 差值: 0.0000000000
数值梯度: 0.00264761, 反向传播: 0.00264761, 差值: 0.0000000000
数值梯度: 0.05817381, 反向传播: 0.05817381, 差值: 0.0000000000
数值梯度: 0.05817381, 反向传播: 0.05817381, 差值: 0.0000000000
数值梯度: 0.05817381, 反向传播: 0.05817381, 差值: 0.0000000000
数值梯度: 0.10676466, 反向传播: 0.10676466, 差值: 0.0000000000
最大差值: 4.4769118967558086e-11
梯度检查通过！


In [1]:
from math import exp
import random


def sigmoid(x):
    return 1.0 / (1.0 + exp(-x))


class Node:
    def __init__(self, layer_index, node_index):
        self.layer_index = layer_index
        self.node_index = node_index
        self.downstream = []
        self.upstream = []
        self.output = 0.0
        self.delta = 0.0

    def set_output(self, output):
        self.output = output

    def append_downstream_connection(self, conn):
        self.downstream.append(conn)

    def append_upstream_connection(self, conn):
        self.upstream.append(conn)

    def calc_output(self):
        weighted_sum = sum(
            conn.upstream_node.output * conn.weight
            for conn in self.upstream
        )
        self.output = sigmoid(weighted_sum)

    def calc_output_layer_delta(self, label):
        self.delta = (
            self.output
            * (1 - self.output)
            * (label - self.output)
        )

    def calc_hidden_layer_delta(self):
        downstream_delta = sum(
            conn.downstream_node.delta * conn.weight
            for conn in self.downstream
        )
        self.delta = (
            self.output
            * (1 - self.output)
            * downstream_delta
        )


class ConstNode:
    def __init__(self, layer_index, node_index):
        self.layer_index = layer_index
        self.node_index = node_index
        self.downstream = []
        self.output = 1.0
        self.delta = 0.0

    def append_downstream_connection(self, conn):
        self.downstream.append(conn)

    def calc_hidden_layer_delta(self):
        downstream_delta = sum(
            conn.downstream_node.delta * conn.weight
            for conn in self.downstream
        )
        self.delta = (
            self.output
            * (1 - self.output)
            * downstream_delta
        )


class Connection:
    def __init__(self, upstream_node, downstream_node):
        self.upstream_node = upstream_node
        self.downstream_node = downstream_node
        self.weight = random.uniform(-0.1, 0.1)
        self.gradient = 0.0

    def calc_gradient(self):
        self.gradient = (
            self.downstream_node.delta
            * self.upstream_node.output
        )

    def get_gradient(self):
        return self.gradient

    def update_weight(self, rate):
        self.calc_gradient()
        self.weight += rate * self.gradient

    def __str__(self):
        return (
            f"({self.upstream_node.layer_index}-"
            f"{self.upstream_node.node_index}) -> "
            f"({self.downstream_node.layer_index}-"
            f"{self.downstream_node.node_index}) = "
            f"{self.weight:.6f}"
        )


class Layer:
    def __init__(self, layer_index, node_count):
        self.layer_index = layer_index
        self.nodes = []

        for i in range(node_count):
            self.nodes.append(Node(layer_index, i))

        self.nodes.append(ConstNode(layer_index, node_count))

    def set_output(self, data):
        for i in range(len(data)):
            self.nodes[i].set_output(data[i])

    def calc_output(self):
        for node in self.nodes[:-1]:
            node.calc_output()

    def dump(self):
        for node in self.nodes:
            print(
                f"{node.layer_index}-{node.node_index}: "
                f"output={node.output}, delta={node.delta}"
            )


class Connections:
    def __init__(self):
        self.connections = []

    def add_connection(self, connection):
        self.connections.append(connection)

    def dump(self):
        for conn in self.connections:
            print(conn)


class Network:
    def __init__(self, layers):
        self.connections = Connections()
        self.layers = []

        layer_count = len(layers)

        for i in range(layer_count):
            self.layers.append(Layer(i, layers[i]))

        for layer in range(layer_count - 1):
            for upstream_node in self.layers[layer].nodes:
                for downstream_node in self.layers[layer + 1].nodes[:-1]:
                    conn = Connection(upstream_node, downstream_node)
                    self.connections.add_connection(conn)
                    upstream_node.append_downstream_connection(conn)
                    downstream_node.append_upstream_connection(conn)

    def predict(self, sample):
        self.layers[0].set_output(sample)

        for i in range(1, len(self.layers)):
            self.layers[i].calc_output()

        return [node.output for node in self.layers[-1].nodes[:-1]]

    def calc_delta(self, label):
        output_nodes = self.layers[-1].nodes

        for i in range(len(label)):
            output_nodes[i].calc_output_layer_delta(label[i])

        for layer in self.layers[-2::-1]:
            for node in layer.nodes:
                node.calc_hidden_layer_delta()

    def update_weight(self, rate):
        for layer in self.layers[:-1]:
            for node in layer.nodes:
                for conn in node.downstream:
                    conn.update_weight(rate)

    def train_one_sample(self, label, sample, rate):
        self.predict(sample)
        self.calc_delta(label)
        self.update_weight(rate)

    def train(self, labels, data_set, rate, iteration):
        for i in range(iteration):
            for d in range(len(data_set)):
                self.train_one_sample(labels[d], data_set[d], rate)

    # 保留前面梯度检查用到的两个方法
    def calc_gradient(self):
        for layer in self.layers[:-1]:
            for node in layer.nodes:
                for conn in node.downstream:
                    conn.calc_gradient()

    def get_gradient(self, label, sample):
        self.predict(sample)
        self.calc_delta(label)
        self.calc_gradient()


class Loader:
    def __init__(self, path, count):
        self.path = path
        self.count = count

    def get_file_content(self):
        with open(self.path, "rb") as f:
            content = f.read()
        return content

    def to_int(self, byte):
        return byte


class ImageLoader(Loader):
    def get_picture(self, content, index):
        start = index * 28 * 28 + 16
        picture = []

        for i in range(28):
            picture.append([])
            for j in range(28):
                picture[i].append(
                    self.to_int(content[start + i * 28 + j])
                )

        return picture

    def get_one_sample(self, picture):
        sample = []
        for i in range(28):
            for j in range(28):
                sample.append(picture[i][j] / 255.0)
        return sample

    def load(self):
        content = self.get_file_content()
        data_set = []

        for index in range(self.count):
            data_set.append(
                self.get_one_sample(
                    self.get_picture(content, index)
                )
            )

        return data_set


class LabelLoader(Loader):
    def load(self):
        content = self.get_file_content()
        labels = []

        for index in range(self.count):
            labels.append(self.norm(content[index + 8]))

        return labels

    def norm(self, label):
        label_vec = []
        label_value = self.to_int(label)

        for i in range(10):
            if i == label_value:
                label_vec.append(0.9)
            else:
                label_vec.append(0.1)

        return label_vec


def get_training_data_set(count=60000):
    image_loader = ImageLoader(
        "data/train-images-idx3-ubyte",
        count
    )
    label_loader = LabelLoader(
        "data/train-labels-idx1-ubyte",
        count
    )
    return image_loader.load(), label_loader.load()


def get_test_data_set(count=10000):
    image_loader = ImageLoader(
        "data/t10k-images-idx3-ubyte",
        count
    )
    label_loader = LabelLoader(
        "data/t10k-labels-idx1-ubyte",
        count
    )
    return image_loader.load(), label_loader.load()


def get_result(vec):
    max_value_index = 0
    max_value = 0

    for i in range(len(vec)):
        if vec[i] > max_value:
            max_value = vec[i]
            max_value_index = i

    return max_value_index


def evaluate(network, test_data_set, test_labels):
    error = 0
    total = len(test_data_set)

    for i in range(total):
        label = get_result(test_labels[i])
        predict = get_result(network.predict(test_data_set[i]))
        if label != predict:
            error += 1

    return float(error) / float(total)


def train_and_evaluate():
    train_data_set, train_labels = get_training_data_set(200)
    test_data_set, test_labels = get_test_data_set(50)

    network = Network([784, 20, 10])

    for epoch in range(1, 4):
        network.train(
            train_labels,
            train_data_set,
            0.3,
            1
        )

        error_ratio = evaluate(
            network,
            test_data_set,
            test_labels
        )

        print(
            f"第 {epoch} 轮训练完成，"
            f"错误率：{error_ratio:.2%}，"
            f"准确率：{1 - error_ratio:.2%}"
        )

    real_number = get_result(test_labels[0])
    predicted_number = get_result(
        network.predict(test_data_set[0])
    )

    print("第一张测试图片的真实数字:", real_number)
    print("神经网络预测的数字:", predicted_number)

    return network


if __name__ == "__main__":
    network = train_and_evaluate()


第 1 轮训练完成，错误率：90.00%，准确率：10.00%
第 2 轮训练完成，错误率：66.00%，准确率：34.00%
第 3 轮训练完成，错误率：40.00%，准确率：60.00%
第一张测试图片的真实数字: 7
神经网络预测的数字: 7


In [2]:

import numpy as np
from pathlib import Path

if "__file__" in globals():
    BASE_DIR = Path(__file__).resolve().parent
else:
    BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data"


class SigmoidActivator:
    def forward(self, weighted_input):
        return 1.0 / (1.0 + np.exp(-weighted_input))

    def backward(self, output):
        return output * (1.0 - output)

class FullConnectedLayer:
    def __init__(self, input_size, output_size, activator):
        self.input_size = input_size
        self.output_size = output_size
        self.activator = activator

        self.W = np.random.uniform(
            -0.1, 0.1,
            (output_size, input_size)
        )

        self.b = np.zeros((output_size, 1))
        self.output = np.zeros((output_size, 1))

    def forward(self, input_array):
        self.input = input_array

        self.output = self.activator.forward(
            np.dot(self.W, input_array) + self.b
        )

    def backward(self, delta_array):
        self.delta = (
            self.activator.backward(self.input)
            * np.dot(self.W.T, delta_array)
        )

        self.W_grad = np.dot(
            delta_array,
            self.input.T
        )

        self.b_grad = delta_array

    def update(self, learning_rate):
        self.W += learning_rate * self.W_grad
        self.b += learning_rate * self.b_grad

class Network:
    def __init__(self, layers):
        self.layers = []

        for i in range(len(layers) - 1):
            self.layers.append(
                FullConnectedLayer(
                    layers[i],
                    layers[i + 1],
                    SigmoidActivator()
                )
            )

    def predict(self, sample):
        output = sample

        for layer in self.layers:
            layer.forward(output)
            output = layer.output

        return output

    def train(self, labels, data_set, rate, epoch):
        for i in range(epoch):
            for d in range(len(data_set)):
                self.train_one_sample(
                    labels[d],
                    data_set[d],
                    rate
                )

    def train_one_sample(self, label, sample, rate):
        self.predict(sample)
        self.calc_gradient(label)
        self.update_weight(rate)

    def calc_gradient(self, label):
        delta = (
            self.layers[-1].activator.backward(
                self.layers[-1].output
            )
            * (label - self.layers[-1].output)
        )

        for layer in self.layers[::-1]:
            layer.backward(delta)
            delta = layer.delta

        return delta

    def update_weight(self, rate):
        for layer in self.layers:
            layer.update(rate)

class Loader:
    def __init__(self, path, count):
        self.path = path
        self.count = count

    def get_file_content(self):
        if not self.path.is_file():
            raise FileNotFoundError(
                f"未找到 MNIST 文件：{self.path}\n"
                "请将4个 .ubyte 文件解压并放进 data 文件夹。"
            )

        with open(self.path, "rb") as f:
            content = f.read()

        return content

    def to_int(self, byte):
        return byte

class ImageLoader(Loader):
    def get_picture(self, content, index):
        start = index * 28 * 28 + 16
        picture = []

        for i in range(28):
            picture.append([])

            for j in range(28):
                picture[i].append(
                    self.to_int(
                        content[start + i * 28 + j]
                    )
                )

        return picture

    def get_one_sample(self, picture):
        sample = []

        for i in range(28):
            for j in range(28):
                sample.append(
                    picture[i][j] / 255.0
                )

        return sample

    def load(self):
        content = self.get_file_content()
        data_set = []

        for index in range(self.count):
            data_set.append(
                self.get_one_sample(
                    self.get_picture(content, index)
                )
            )

        return data_set

class LabelLoader(Loader):
    def load(self):
        content = self.get_file_content()
        labels = []

        for index in range(self.count):
            labels.append(
                self.norm(content[index + 8])
            )

        return labels

    def norm(self, label):
        label_vec = []
        label_value = self.to_int(label)

        for i in range(10):
            if i == label_value:
                label_vec.append(0.9)
            else:
                label_vec.append(0.1)

        return label_vec

def get_training_data_set(count=200):
    image_loader = ImageLoader(
        DATA_DIR / "train-images-idx3-ubyte",
        count
    )

    label_loader = LabelLoader(
        DATA_DIR / "train-labels-idx1-ubyte",
        count
    )

    images = image_loader.load()
    labels = label_loader.load()

    data_set = [
        np.array(image).reshape(784, 1)
        for image in images
    ]

    label_set = [
        np.array(label).reshape(10, 1)
        for label in labels
    ]

    return data_set, label_set

def get_test_data_set(count=50):
    image_loader = ImageLoader(
        DATA_DIR / "t10k-images-idx3-ubyte",
        count
    )

    label_loader = LabelLoader(
        DATA_DIR / "t10k-labels-idx1-ubyte",
        count
    )

    images = image_loader.load()
    labels = label_loader.load()

    data_set = [
        np.array(image).reshape(784, 1)
        for image in images
    ]

    label_set = [
        np.array(label).reshape(10, 1)
        for label in labels
    ]

    return data_set, label_set

def get_result(vec):
    vec = np.asarray(vec).reshape(-1)

    max_value_index = 0
    max_value = 0

    for i in range(len(vec)):
        if vec[i] > max_value:
            max_value = vec[i]
            max_value_index = i

    return max_value_index

def evaluate(network, test_data_set, test_labels):
    error = 0
    total = len(test_data_set)

    for i in range(total):
        label = get_result(test_labels[i])

        predict = get_result(
            network.predict(test_data_set[i])
        )

        if label != predict:
            error += 1

    return float(error) / float(total)

def train_and_evaluate():
    train_data_set, train_labels = get_training_data_set(200)
    test_data_set, test_labels = get_test_data_set(50)

    print("图片形状:", train_data_set[0].shape)
    print("标签形状:", train_labels[0].shape)

    np.random.seed(0)

    network = Network([784, 20, 10])

    for epoch in range(1, 4):
        network.train(
            train_labels,
            train_data_set,
            0.3,
            1
        )

        error_ratio = evaluate(
            network,
            test_data_set,
            test_labels
        )

        print(
            f"第 {epoch} 轮训练完成，"
            f"错误率：{error_ratio:.2%}，"
            f"准确率：{1 - error_ratio:.2%}"
        )

    print("前5张测试图片的识别结果：")

    for i in range(min(5, len(test_data_set))):
        real_number = get_result(
            test_labels[i]
        )

        predicted_number = get_result(
            network.predict(test_data_set[i])
        )

        print(
            f"第 {i + 1} 张："
            f"真实={real_number}，"
            f"预测={predicted_number}"
        )

    return network

if __name__ == "__main__":
    network = train_and_evaluate()

图片形状: (784, 1)
标签形状: (10, 1)
第 1 轮训练完成，错误率：90.00%，准确率：10.00%
第 2 轮训练完成，错误率：62.00%，准确率：38.00%
第 3 轮训练完成，错误率：40.00%，准确率：60.00%
前5张测试图片的识别结果：
第 1 张：真实=7，预测=7
第 2 张：真实=2，预测=2
第 3 张：真实=1，预测=1
第 4 张：真实=0，预测=0
第 5 张：真实=4，预测=7
